In [ ]:
# 根据chunk内容查找对应的数据库IDimport sysimport ossys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))from rag.database import get_db_connectiondef find_chunk_id_by_content(keyword: str, top_k: int = 5):    """    根据关键词查找包含该内容的chunk ID        Args:        keyword: 要搜索的关键词        top_k: 返回匹配结果数量    """    conn = get_db_connection()    cur = conn.execute(        """        SELECT id, chunk_text, source_file         FROM financial_vectors         WHERE chunk_text LIKE %s        ORDER BY id        LIMIT %s        """,        (f"%{keyword}%", top_k * 10)    )        results = cur.fetchall()    conn.close()        if not results:        print(f"未找到包含 '{keyword}' 的chunk")        return []        print(f"找到 {len(results)} 个包含 '{keyword}' 的chunk:")    print("=" * 80)        for i, (chunk_id, text, source) in enumerate(results[:top_k], 1):        # 找到关键词位置并高亮显示        highlight_pos = text.lower().find(keyword.lower())        if highlight_pos != -1:            start = max(0, highlight_pos - 40)            end = min(len(text), highlight_pos + len(keyword) + 40)            preview = text[start:end]            if start > 0:                preview = "..." + preview            if end < len(text):                preview = preview + "..."        else:            preview = text[:100] + "..." if len(text) > 100 else text                 print(f"\n[{i}] ID: {chunk_id} | 来源: {source}")        print(f"    预览: {preview}")        return [r[0] for r in results[:top_k]]# 测试：查找包含"总资产"的chunkmatched_ids = find_chunk_id_by_content("支付给职工及为职工支付的", top_k=10)